In [0]:
CATALOG = "electrocasa_dev"

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {CATALOG}
""")

print(f"Catálogo creado correctamente: {CATALOG}")

In [0]:
display(spark.sql("SHOW CATALOGS"))

In [0]:
CATALOG = "electrocasa_dev"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold
""")

print("Schemas creados correctamente:")
print("- bronze")
print("- silver")
print("- gold")

In [0]:
display(
    spark.sql("""
    SHOW SCHEMAS IN electrocasa_dev
    """)
)

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS
electrocasa_dev.default.vol_landing
""")

print("Volume vol_landing creado correctamente")

In [0]:
display(
    spark.sql("""
    SHOW VOLUMES IN electrocasa_dev.default
    """)
)

In [0]:
landing = "/Volumes/electrocasa_dev/default/vol_landing"

fuentes = [
    "ventas",
    "catalogo",
    "empleados",
    "resenas",
    "devoluciones",
    "sucursales"
]

for fuente in fuentes:
    dbutils.fs.mkdirs(f"{landing}/{fuente}")

print("Carpetas creadas correctamente")

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/electrocasa_dev/default/vol_landing"
    )
)

In [0]:
landing = "/Volumes/electrocasa_dev/default/vol_landing"

fuentes = [
    "ventas",
    "catalogo",
    "empleados",
    "resenas",
    "devoluciones",
    "sucursales"
]

for fuente in fuentes:
    print(f"\n========== {fuente.upper()} ==========")

    archivos = dbutils.fs.ls(f"{landing}/{fuente}")

    if len(archivos) == 0:
        print("SIN ARCHIVOS")
    else:
        for archivo in archivos:
            print(archivo.name)

In [0]:
path_ventas = "/Volumes/electrocasa_dev/default/vol_landing/ventas"

df_ventas_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv(path_ventas)
)

print("COLUMNAS DE VENTAS:")
print(df_ventas_exploracion.columns)

print("\nSCHEMA DE EXPLORACIÓN:")
df_ventas_exploracion.printSchema()

display(df_ventas_exploracion.limit(10))

In [0]:
inferSchema=True

In [0]:
path_empleados = "/Volumes/electrocasa_dev/default/vol_landing/empleados"

df_empleados_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv(path_empleados)
)

print("COLUMNAS DE EMPLEADOS:")
print(df_empleados_exploracion.columns)

print("\nSCHEMA DE EXPLORACIÓN:")
df_empleados_exploracion.printSchema()

display(df_empleados_exploracion.limit(15))

In [0]:
print(
    "Cantidad registros Empleados lote 1:",
    df_empleados_exploracion.count()
)

In [0]:
path_devoluciones = "/Volumes/electrocasa_dev/default/vol_landing/devoluciones"

df_devoluciones_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv(path_devoluciones)
)

print("COLUMNAS DE DEVOLUCIONES:")
print(df_devoluciones_exploracion.columns)

print("\nSCHEMA DE EXPLORACIÓN:")
df_devoluciones_exploracion.printSchema()

display(df_devoluciones_exploracion.limit(15))

In [0]:
print(
    "Cantidad registros Devoluciones lote 1:",
    df_devoluciones_exploracion.count()
)

In [0]:
path_catalogo = "/Volumes/electrocasa_dev/default/vol_landing/catalogo"

df_catalogo_raw = spark.read.text(path_catalogo)

print("PRIMERAS LINEAS DEL CATALOGO:")
display(df_catalogo_raw.limit(10))

In [0]:
print(
    "Cantidad de líneas del archivo Catálogo:",
    df_catalogo_raw.count()
)

In [0]:
df_catalogo_exploracion = (
    spark.read
    .option("multiLine", "true")
    .json(path_catalogo)
)

print("COLUMNAS DEL CATALOGO:")
print(df_catalogo_exploracion.columns)

print("\nSCHEMA DEL CATALOGO:")
df_catalogo_exploracion.printSchema()

display(df_catalogo_exploracion.limit(15))

In [0]:
path_resenas = "/Volumes/electrocasa_dev/default/vol_landing/resenas"

df_resenas_raw = spark.read.text(path_resenas)

print("PRIMERAS LINEAS RAW DE RESENAS:")
display(df_resenas_raw.limit(5))

In [0]:
print(
    "Cantidad registros/lineas Reseñas lote 1:",
    df_resenas_raw.count()
)

In [0]:
df_resenas_exploracion = (
    spark.read
    .json(path_resenas)
)

print("COLUMNAS DE RESENAS:")
print(df_resenas_exploracion.columns)

print("\nSCHEMA DE RESENAS:")
df_resenas_exploracion.printSchema()

display(df_resenas_exploracion.limit(15))

In [0]:
path_sucursales = "/Volumes/electrocasa_dev/default/vol_landing/sucursales"

df_sucursales_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv(path_sucursales)
)

print("COLUMNAS DE SUCURSALES:")
print(df_sucursales_exploracion.columns)

print("\nSCHEMA DE EXPLORACIÓN:")
df_sucursales_exploracion.printSchema()

display(df_sucursales_exploracion.limit(20))

In [0]:
print(
    "Cantidad registros Sucursales:",
    df_sucursales_exploracion.count()
)

In [0]:
print("========================================")
print("RESUMEN EXPLORACION - ELECTROCASA LOTE 1")
print("========================================")

fuentes_exploradas = {
    "ventas": df_ventas_exploracion,
    "empleados": df_empleados_exploracion,
    "devoluciones": df_devoluciones_exploracion,
    "sucursales": df_sucursales_exploracion,
    "catalogo": df_catalogo_exploracion,
    "resenas": df_resenas_exploracion
}

for nombre, df in fuentes_exploradas.items():
    print(f"\n===== {nombre.upper()} =====")
    print(f"Registros: {df.count()}")
    print("Columnas:")
    
    for columna in df.columns:
        print(f" - {columna}")

In [0]:
# ========================================
# RECREAR DATAFRAMES DE EXPLORACION
# ========================================

# VENTAS
df_ventas_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv("/Volumes/electrocasa_dev/default/vol_landing/ventas")
)

# EMPLEADOS
df_empleados_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv("/Volumes/electrocasa_dev/default/vol_landing/empleados")
)

# DEVOLUCIONES
df_devoluciones_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv("/Volumes/electrocasa_dev/default/vol_landing/devoluciones")
)

# SUCURSALES
df_sucursales_exploracion = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .csv("/Volumes/electrocasa_dev/default/vol_landing/sucursales")
)

# CATALOGO
df_catalogo_exploracion = (
    spark.read
    .option("multiLine", "true")
    .json("/Volumes/electrocasa_dev/default/vol_landing/catalogo")
)

# RESENAS - JSON Lines
df_resenas_exploracion = (
    spark.read
    .json("/Volumes/electrocasa_dev/default/vol_landing/resenas")
)

print("DataFrames de exploración recreados correctamente.")

In [0]:
print("========================================")
print("RESUMEN EXPLORACION - ELECTROCASA LOTE 1")
print("========================================")

fuentes_exploradas = {
    "ventas": df_ventas_exploracion,
    "empleados": df_empleados_exploracion,
    "devoluciones": df_devoluciones_exploracion,
    "sucursales": df_sucursales_exploracion,
    "catalogo": df_catalogo_exploracion,
    "resenas": df_resenas_exploracion
}

for nombre, df in fuentes_exploradas.items():
    print(f"\n===== {nombre.upper()} =====")
    print(f"Registros: {df.count()}")
    print("Columnas:")

    for columna in df.columns:
        print(f" - {columna}")

In [0]:
from pyspark.sql import Row

conteos_lote1 = []

for nombre, df in fuentes_exploradas.items():
    conteos_lote1.append(
        Row(
            fuente=nombre,
            lote="lote_01",
            registros=df.count()
        )
    )

df_conteos_lote1 = spark.createDataFrame(conteos_lote1)

display(df_conteos_lote1)

In [0]:
from pyspark.sql import Row

conteos_lote1 = []

for nombre, df in fuentes_exploradas.items():
    conteos_lote1.append(
        Row(
            fuente=nombre,
            lote="lote_01",
            registros=df.count()
        )
    )

df_conteos_lote1 = spark.createDataFrame(conteos_lote1)

display(df_conteos_lote1)

In [0]:
df_conteos_lote1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("electrocasa_dev.default.evidencia_conteos_lote1")

print("Tabla creada correctamente")

In [0]:
%sql
SELECT *
FROM electrocasa_dev.default.evidencia_conteos_lote1;